# Clean the merged floorsheet data

Loads `data/processed/floorsheet.parquet`, checks for nulls/duplicates/bad values, fixes what can be fixed, and writes `data/processed/floorsheet_clean.parquet`.

Load the merged data.

In [1]:
from pathlib import Path
import pandas as pd

IN_PATH = Path("../data/processed/floorsheet.parquet")
OUT_PATH = Path("../data/processed/floorsheet_clean.parquet")

df = pd.read_parquet(IN_PATH)
df.shape

(47346003, 8)

Check for missing values.

In [2]:
df.isna().sum()

transaction    0
symbol         0
buyer          0
seller         0
quantity       0
rate           0
amount         0
date           0
dtype: int64

Check for duplicate rows and duplicate transaction ids.

In [4]:
n_dup_rows = df.duplicated().sum()
print("full duplicate rows:", n_dup_rows)

full duplicate rows: 0


In [5]:
n_dup_txn = df["transaction"].duplicated().sum()
print("duplicate transaction ids:", n_dup_txn)

duplicate transaction ids: 0


Check for invalid quantity / rate / amount (zero or negative).

In [7]:
n_bad_qty = (df["quantity"] <= 0).sum()
n_bad_rate = (df["rate"] <= 0).sum()
n_bad_amount = (df["amount"] <= 0).sum()
print("quantity less than 0:", n_bad_qty)
print("rate less than 0:", n_bad_rate)
print("amount less than 0:", n_bad_amount)

quantity less than 0: 0
rate less than 0: 2
amount less than 0: 2


Check that amount actually equals quantity * rate.

In [8]:
mismatch = (df["quantity"] * df["rate"] - df["amount"]).abs()
n_mismatch = (mismatch > 1).sum()
print("rows where amount != quantity * rate (off by > 1):", n_mismatch)
mismatch.describe()

rows where amount != quantity * rate (off by > 1): 0


count    4.734600e+07
mean     1.691623e-04
std      1.002346e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.500000e-01
dtype: float64

Check for self-trades (same broker on both sides).

In [9]:
n_self_trades = (df["buyer"] == df["seller"]).sum()
print("self trades (buyer == seller):", n_self_trades)

self trades (buyer == seller): 1256268


Drop exact duplicate rows and any rows with non-positive quantity/rate/amount.

In [10]:
before = len(df)

df_clean = df.drop_duplicates()
df_clean = df_clean[
    (df_clean["quantity"] > 0) & (df_clean["rate"] > 0) & (df_clean["amount"] > 0)
]
df_clean = df_clean.reset_index(drop=True)

after = len(df_clean)
n_dropped = before - after
print(f"dropped {n_dropped:,} rows ({before:,} -> {after:,})")

dropped 2 rows (47,346,003 -> 47,346,001)


Sanity check the cleaned data.

In [11]:
df_clean.info()
df_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 47346001 entries, 0 to 47346000
Data columns (total 8 columns):
 #   Column       Dtype         
---  ------       -----         
 0   transaction  string        
 1   symbol       string        
 2   buyer        string        
 3   seller       string        
 4   quantity     float64       
 5   rate         float64       
 6   amount       float64       
 7   date         datetime64[us]
dtypes: datetime64[us](1), float64(3), string(4)
memory usage: 3.9 GB


,transaction,symbol,buyer,seller,quantity,rate,amount,date
0,2024010103010653,GBBL,22,53,50.0,410.0,20500.0,2024-01-01
1,2024010105005330,NIBLPF,26,10,5647.0,9.3,52517.1,2024-01-01
2,2024010104015466,PRIN,20,34,10.0,851.0,8510.0,2024-01-01
3,2024010101071456,GVL,3,34,200.0,418.5,83700.0,2024-01-01
4,2024010103010652,HLI,39,34,10.0,438.0,4380.0,2024-01-01


`df_clean` was already deduplicated and filtered a few cells back — this step doesn't do any cleaning itself.
It just writes that already-cleaned DataFrame to `floorsheet_clean.parquet` so later notebooks can load it directly.

In [12]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(OUT_PATH, index=False)
n_written = len(df_clean)
print(f"Wrote {n_written:,} rows to {OUT_PATH}")

Wrote 47,346,001 rows to ../data/processed/floorsheet_clean.parquet
